In [1]:
pip install transformers datasets torch pandas scikit-learn accelerate optuna wandb

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 72.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/6 [sentry-sdk]  WARNING: The script mako-render is installed in '/home/dja1/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [wandb]  WARNING: The scripts wandb and wb are installed in '/home/dja1/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [optuna]  WARNING: The script optuna is installed in '/home/dja1/.local/bin' which is not on PATH.
  Consider adding this direct

In [1]:
MODEL_NAME = "GroNLP/hateBERT"

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "GroNLP/hateBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True,
    id2label={0: "Non-Toxic", 1: "Insults and Flaming", 2: "Other Offensive Texts", 
              3: "Hate and Harassment 	", 4: "Threats", 5: "Extremism"},
    label2id={"Non-Toxic": 0, "Insults and Flaming": 1, "Other Offensive Texts": 2,
              "Hate and Harassment 	": 3, "Threats": 4, "Extremism": 5}
    )

In [3]:
from pathlib import Path

main_dir = Path.cwd()

# Using GameTox

In [4]:
PATH_GAMETOX_TRAIN = "Existing_Datasets/GameTox/train.csv"
PATH_GAMETOX_VAL = "Existing_Datasets/GameTox/val.csv"

In [5]:
import pandas as pd

df = pd.read_csv( main_dir / PATH_GAMETOX_TRAIN )
df_2 = pd.read_csv( main_dir / PATH_GAMETOX_VAL )

In [6]:
df.head()

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0


In [7]:
df_2.tail()

,index,message,label
5362,52888,u are afk bot,1.0
5363,14736,gsor please learn alternative scouting positio...,1.0
5364,15844,i fucking love putin,5.0
5365,20533,leo go and kill,0.0
5366,17465,겁먹은 **끼들 처모여 있는 것처럼 꼴 좋네,0.0


In [8]:
df["label"] = df["label"].astype(int)
df_2["label"] = df_2["label"].astype(int)
df = df[["message", "label"]]
df_2 = df_2[["message", "label"]]

In [9]:
from datasets import Dataset
from transformers import set_seed

set_seed(42)

dataset_train = Dataset.from_pandas(df)
dataset_val = Dataset.from_pandas(df_2)

In [10]:
def tokenize(batch):

    return tokenizer(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [11]:
tokenized_dataset_train = dataset_train.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/42959 [00:00<?, ? examples/s]

In [12]:
tokenized_dataset_val = dataset_val.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/5367 [00:00<?, ? examples/s]

# Optuna Storage for Hyperparameter Search

In [13]:
import optuna
from optuna.storages import RDBStorage

# Define persistent storage
storage = RDBStorage("sqlite:///optuna_trials.db")

study = optuna.create_study(
    study_name="HateBERT_optuna_study",
    direction="maximize",
    storage=storage,
    load_if_exists=True
)

[I 2026-08-08 13:35:24,497] Using an existing study with name 'HateBERT_optuna_study' instead of creating a new one.


In [25]:
from sklearn.metrics import average_precision_score, f1_score
import numpy as np

def softmax(x, axis=1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def compute_objective(metrics):
    return metrics["eval_macro_F1"]

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    probabilities = softmax(
        logits, 
        axis=1
    )

    n_classes = logits.shape[1]

    ap_scores = []
    for i in range(n_classes):
        y_true = (labels == i).astype(int)
        y_score = probabilities[:, i]
        ap = average_precision_score(y_true, y_score)
        ap_scores.append(ap)

    return {
        "macro_AUPRC": np.mean(ap_scores),

        "macro_F1":
            f1_score(
                labels,
                predictions,
                average="macro"
            )
    }

In [ ]:
import wandb

wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/dja1/.netrc
wandb: Currently logged in as: dj125101308 (dj125101308-university-college-cork) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
from transformers import TrainingArguments

wandb.init(project="hf-optuna", name="HateBERT_optuna_study")

training_args = TrainingArguments(
    output_dir="./HateBERT-GameTox",
    logging_dir="./HateBERT-GameTox-logs",

    num_train_epochs=3,
    logging_strategy="epoch",

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="wandb",
    run_name="HateBERT_optuna_study"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
from transformers import Trainer


trainer = Trainer(
    model_init=model_init,

    args=training_args,

    train_dataset=
        tokenized_dataset_train,

    eval_dataset=
        tokenized_dataset_val,

    compute_metrics=
        compute_metrics,

    processing_class=
        tokenizer
)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [16, 32, 64]
        ),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
    }

best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=10,
    compute_objective=compute_objective,
    study_name="HateBERT_optuna_study",
    storage="sqlite:///optuna_trials.db",
    load_if_exists=True
)

[I 2026-08-08 14:18:08,106] Using an existing study with name 'HateBERT_optuna_study' instead of creating a new one.
[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro Auprc,Macro F1
1,0.610148,0.387763,0.346104,0.270070
2,0.371152,0.361131,0.355114,0.319240
3,0.350746,0.356496,0.357603,0.320815


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Macro Auprc,Macro F1
1,0.439245,0.334980,0.370280,0.347816
2,0.304911,0.323815,0.388259,0.352771
3,0.283484,0.326032,0.398733,0.354257


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-08-08 19:50:36,100] Trial 11 finished with value: 0.3542574821297668 and parameters: {'learning_rate': 7.824356947225773e-06, 'per_device_train_batch_size': 64, 'weight_decay': 0.1464162997538629}. Best is trial 5 with value: 0.48395720895214467.


In [30]:
print(best_run)

BestRun(run_id='5', objective=0.48395720895214467, hyperparameters={'learning_rate': 2.717794860748917e-05, 'per_device_train_batch_size': 32, 'weight_decay': 0.2281571823779551}, run_summary=None)


In [64]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [78]:
def predict_toxicity(text):

    inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
    )

    inputs = {
    key: value.to(device)
    for key, value in inputs.items()
    }


    with torch.no_grad():
        output = model(**inputs)


    probabilities = torch.softmax(
    output.logits,
    dim=1
    )

    pred = torch.argmax(
    probabilities,
    dim=1
    ).item()

    return probabilities, pred

In [86]:
model.to(device)
model.eval()

text = "you are bad"

probabilities, pred = predict_toxicity(text)

print(probabilities)
print(model.config.id2label[pred])

tensor([[1.6374e-02, 9.6471e-01, 1.6565e-02, 1.4334e-03, 5.4496e-04, 3.6862e-04]],
       device='cuda:0')
Insults and Flaming


In [71]:
df_test_set = pd.read_csv(main_dir.parent.parent / "WebScraper" / "text_data" / "Steam" / "Counter-Strike_2_labeled.csv")

In [87]:
df_test_set["label"].value_counts(normalize=True)

label
0    0.857685
2    0.079696
1    0.055028
3    0.003795
4    0.003795
Name: proportion, dtype: float64

In [81]:
model = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "hateBERT-GameTox" / "checkpoint-6444",
    id2label={0: "Non-Toxic", 1: "Insults and Flaming", 2: "Other Offensive Texts", 
              3: "Hate and Harassment 	", 4: "Threats", 5: "Extremism"},
    label2id={"Non-Toxic": 0, "Insults and Flaming": 1, "Other Offensive Texts": 2,
              "Hate and Harassment 	": 3, "Threats": 4, "Extremism": 5}
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3701.44it/s]


In [83]:
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [84]:
label_list = []

for text in df_test_set["text"]:

    probabilities, pred = predict_toxicity(text)
    label_list.append(model.config.id2label[pred])


In [85]:
label_list = pd.Series(label_list)
label_list.value_counts()

Non-Toxic                400
Other Offensive Texts     64
Insults and Flaming       63
Name: count, dtype: int64